In [1]:
pip install datasets transformers evaluate torch joblib scikit-learn accelerate seqeval

Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from datasets import load_dataset
import evaluate
import torch
import numpy as np

/home/ankushchaudhary1/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset("conll2003")

In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForTokenClassification.from_pretrained("bert-base-uncased", num_labels=9)  # 9 labels in CoNLL-2003

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, padding="max_length", max_length=128, is_split_into_words=True)
    
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # List of word IDs per tokenized example
        label_ids = [-100 if word_id is None else label[word_id] for word_id in word_ids]  # Align labels to tokens
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [6]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

Map: 100%|██████████| 3250/3250 [00:00<00:00, 9199.54 examples/s] 


In [7]:
metric = evaluate.load("seqeval")

labels_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
label_map = {i: label for i, label in enumerate(labels_list)}

In [8]:
def compute_metrics(p):
    logits, labels = p

    if isinstance(logits, np.ndarray):
        logits = torch.tensor(logits)

    predictions = torch.argmax(logits, dim=-1)
    
    if isinstance(labels, np.ndarray):
        labels = torch.tensor(labels)

    labels = labels.detach().cpu().numpy()  # Move to CPU if tensor
    predictions = predictions.detach().cpu().numpy()  # Move to CPU if tensor
    
    predictions_str = [
        [label_map[label] if label != -100 else "O" for label in seq]  # Skip -100 labels
        for seq in predictions
    ]
    labels_str = [
        [label_map[label] if label != -100 else "O" for label in seq]  # Skip -100 labels
        for seq in labels
    ]

    return metric.compute(predictions=predictions_str, references=labels_str)

In [9]:
training_args = TrainingArguments(
    output_dir="./results_ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

/localscratch-nvme/88879/ipykernel_3484190/2631621073.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [10]:
trainer.train()

Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy
1,0.163000,0.060748,"{'precision': 0.14178351093662367, 'recall': 0.9656226126814362, 'f1': 0.24726134585289514, 'number': 2618}","{'precision': 0.1503873613853866, 'recall': 0.8042242079610074, 'f1': 0.2533913488610187, 'number': 1231}","{'precision': 0.08535215065845655, 'recall': 0.9236381322957199, 'f1': 0.15626414318041557, 'number': 2056}","{'precision': 0.1363401456424712, 'recall': 0.9580719709475074, 'f1': 0.23871020811055352, 'number': 3029}",0.122434,0.931162,0.216412,0.839329
2,0.044500,0.055421,"{'precision': 0.21365434800909167, 'recall': 0.9694423223834988, 'f1': 0.35014140856728976, 'number': 2618}","{'precision': 0.17495426575752537, 'recall': 0.8545897644191714, 'f1': 0.2904472667034788, 'number': 1231}","{'precision': 0.0891812865497076, 'recall': 0.9197470817120622, 'f1': 0.16259673258813415, 'number': 2056}","{'precision': 0.12924014805138254, 'recall': 0.9798613403763619, 'f1': 0.2283603908594291, 'number': 3029}",0.136140,0.945713,0.238017,0.851911
3,0.025600,0.056261,"{'precision': 0.23125457205559619, 'recall': 0.9660045836516424, 'f1': 0.37317397078353254, 'number': 2618}","{'precision': 0.17825644504748983, 'recall': 0.8537774167343624, 'f1': 0.2949347551564473, 'number': 1231}","{'precision': 0.09984792070900414, 'recall': 0.9260700389105059, 'f1': 0.1802603550295858, 'number': 2056}","{'precision': 0.14612232808411907, 'recall': 0.9772202046880158, 'f1': 0.2542300094477368, 'number': 3029}",0.150361,0.945153,0.259448,0.866286


TrainOutput(global_step=2634, training_loss=0.0777025925180816, metrics={'train_runtime': 1868.7268, 'train_samples_per_second': 22.541, 'train_steps_per_second': 1.41, 'total_flos': 2751824963545344.0, 'train_loss': 0.0777025925180816, 'epoch': 3.0})

In [11]:
model.save_pretrained("model_taskB_ner")
tokenizer.save_pretrained("model_taskB_ner")

('model_taskB_ner/tokenizer_config.json',
 'model_taskB_ner/special_tokens_map.json',
 'model_taskB_ner/vocab.txt',
 'model_taskB_ner/added_tokens.json',
 'model_taskB_ner/tokenizer.json')